# 한국투자증권 API — 국내 주식 자동매매

**목표:** KIS API로 KOSPI 주식 자동 주문 연결

**순서:**
1. API 키 설정 & 토큰 발급
2. 계좌 잔고 조회
3. 현재가 조회
4. 매수/매도 주문
5. 전략 신호 연결
6. 자동 리밸런싱
7. 일일 모니터링

---
## Cell 1 — API 키 설정

In [2]:
import requests
import json
import pandas as pd
import numpy as np
import datetime
import time
import pickle
import os
import warnings
warnings.filterwarnings('ignore')

# ── API 키 설정 ───────────────────────────────────────────
# KIS Developers에서 발급받은 키
APP_KEY    = 'PSqEVttQjx45twP01RB2zqgnKyAsaKGpt8qL'     # ← 여기에 입력
APP_SECRET = 'iS05IY2Wv9ZgtjuRyb/27b9/WnZhFHANkCIhn/nuOvOLkDiDnn2xeWJV3X4X5w0Kpo8yIU6d71FPWuVg2Z+EqyE9bT3pHpNgR9dK1nzOQ4bKwimoaJASoQ651KF57xyUMaVhrYYcRxSwSUZ6ttZUH0q5fg7zSXGZaYfGiUXHmrkxNYeox+0='  # ← 여기에 입력
ACCOUNT_NO = '46021044'     # ← 계좌번호 앞 8자리 (예: 50123456)
ACCOUNT_CD = '01'               # ← 계좌 상품코드 (보통 01)

# 실전투자 URL
BASE_URL = 'https://openapi.koreainvestment.com:9443'

# 투자 설정
INVEST_AMOUNT = 3_000_000  # 100만원
N_STOCKS      = 3         # 편입 종목 수 (100만원이면 10종목 적당)
                           # 종목당 약 10만원

print('설정 완료')
print(f'APP_KEY    : {APP_KEY[:10]}...')
print(f'ACCOUNT_NO : {ACCOUNT_NO}')
print(f'투자금액   : {INVEST_AMOUNT:,}원')
print(f'편입종목수 : {N_STOCKS}개')
print(f'종목당금액 : {INVEST_AMOUNT//N_STOCKS:,}원')

설정 완료
APP_KEY    : PSqEVttQjx...
ACCOUNT_NO : 46021044
투자금액   : 3,000,000원
편입종목수 : 3개
종목당금액 : 1,000,000원


---
## Cell 2 — 액세스 토큰 발급

In [3]:
def get_access_token():
    """액세스 토큰 발급 (24시간 유효)"""
    url  = f'{BASE_URL}/oauth2/tokenP'
    body = {
        'grant_type': 'client_credentials',
        'appkey'    : APP_KEY,
        'appsecret' : APP_SECRET
    }
    resp = requests.post(url, json=body)
    if resp.status_code == 200:
        token = resp.json()['access_token']
        print(f'토큰 발급 성공: {token[:20]}...')
        return token
    else:
        print(f'토큰 발급 실패: {resp.status_code}')
        print(resp.text[:200])
        return None


def get_headers(tr_id, tr_cont=''):
    """공통 헤더 생성"""
    return {
        'Content-Type' : 'application/json',
        'authorization': f'Bearer {ACCESS_TOKEN}',
        'appkey'       : APP_KEY,
        'appsecret'    : APP_SECRET,
        'tr_id'        : tr_id,
        'tr_cont'      : tr_cont,
        'custtype'     : 'P'
    }


ACCESS_TOKEN = get_access_token()

토큰 발급 성공: eyJ0eXAiOiJKV1QiLCJh...


---
## Cell 3 — 계좌 잔고 조회

In [20]:
def get_balance():
    """
    국내 주식 잔고 조회
    보유 종목 + 예수금 확인
    """
    url    = f'{BASE_URL}/uapi/domestic-stock/v1/trading/inquire-balance'
    params = {
        'CANO'             : ACCOUNT_NO,
        'ACNT_PRDT_CD'     : ACCOUNT_CD,
        'AFHR_FLPR_YN'     : 'N',
        'OFL_YN'           : '',
        'INQR_DVSN'        : '02',
        'UNPR_DVSN'        : '01',
        'FUND_STTL_ICLD_YN': 'N',
        'FNCG_AMT_AUTO_RDPT_YN': 'N',
        'PRCS_DVSN'        : '01',
        'CTX_AREA_FK100'   : '',
        'CTX_AREA_NK100'   : ''
    }
    resp = requests.get(
        url,
        headers=get_headers('TTTC8434R'),
        params=params
    )

    if resp.status_code == 200:
        data = resp.json()
        if data['rt_cd'] == '0':
            output2 = data['output2'][0] if data['output2'] else {}

            # 예수금
            deposit = int(output2.get('dnca_tot_amt', 0))
            total   = int(output2.get('tot_evlu_amt', 0))
            print('계좌 잔고 조회 성공')
            print(f'  예수금 (현금)  : {deposit:>15,}원')
            print(f'  총 평가금액    : {total:>15,}원')

            # 보유 종목
            holdings = data['output1']
            current  = {}
            if holdings:
                print(f'\n  보유 종목 ({len(holdings)}개):')
                print(f'  {"티커":>8} | {"종목명":>12} | {"수량":>6} | '
                      f'{"평균단가":>10} | {"현재가":>10} | {"손익률":>7}')
                print('  ' + '-' * 65)
                for h in holdings:
                    code    = h['pdno']
                    name    = h['prdt_name']
                    qty     = int(h['hldg_qty'])
                    avg_prc = int(float(h['pchs_avg_pric']))
                    cur_prc = int(h['prpr'])
                    pnl_pct = float(h['evlu_pfls_rt'])
                    current[code] = qty
                    print(f'  {code:>8} | {name:>12} | {qty:>6} | '
                          f'{avg_prc:>10,} | {cur_prc:>10,} | '
                          f'{pnl_pct:>6.2f}%')
            else:
                print('\n  보유 종목 없음')

            return deposit, total, current
        else:
            print(f'조회 실패: {data["msg1"]}')
    else:
        print(f'HTTP 오류: {resp.status_code}')
        print(resp.text[:300])
    return 0, 0, {}


deposit, total, current_holdings = get_balance()

계좌 잔고 조회 성공
  예수금 (현금)  :       3,254,398원
  총 평가금액    :       3,259,299원

  보유 종목 (1개):
        티커 |          종목명 |     수량 |       평균단가 |        현재가 |     손익률
  -----------------------------------------------------------------
    073240 |        금호타이어 |      0 |          0 |      4,915 |   0.00%


---
## Cell 4 — 현재가 조회

In [13]:
# 티커 형식 수정 (5930.0 → 005930)
signal_df = pd.read_csv('data/processed/kr_latest_signal.csv')
signal_df['ticker'] = signal_df['ticker'].apply(
    lambda x: str(int(float(x))).zfill(6)
)
top10 = signal_df.nlargest(N_STOCKS, 'score')

print(f'매수 후보 {N_STOCKS}개 종목 현재가 조회')
print(f'  {"티커":>8} | {"종목명":>15} | {"현재가":>10} | {"등락률":>7}')
print('  ' + '-' * 50)

price_dict = {}
for _, row in top10.iterrows():
    code = row['ticker']
    price, change, pct, name = get_price(code)
    if price:
        mark = '▲' if pct > 0 else '▼' if pct < 0 else '-'
        print(f'  {code:>8} | {name:>15} | '
              f'{price:>10,} | {mark}{abs(pct):>5.2f}%')
        price_dict[code] = price
    else:
        print(f'  {code:>8} | 조회 실패')
    time.sleep(0.1)

# 100만원 이하로 살 수 있는 종목만 필터링
target_per = INVEST_AMOUNT // N_STOCKS  # 100만원
print(f'\n종목당 투자금: {target_per:,}원')
print(f'\n살 수 있는 종목:')

affordable = {}
for code, price in price_dict.items():
    if price <= target_per:
        affordable[code] = price
        print(f'  {code}: {price:,}원 ✅')
    else:
        print(f'  {code}: {price:,}원 ❌ (초과)')

# 부족한 종목 수만큼 다음 순위에서 채우기
n_need = N_STOCKS - len(affordable)
print(f'\n추가로 {n_need}개 종목 필요')

if n_need > 0:
    # 4등부터 순서대로 추가
    remaining = signal_df.nlargest(20, 'score')
    remaining = remaining[
        ~remaining['ticker'].isin(list(affordable.keys()) + 
        list(price_dict.keys()))
    ]
    
    for _, row in remaining.iterrows():
        if n_need <= 0:
            break
        code = row['ticker']
        price, _, _, name = get_price(code)
        time.sleep(0.1)
        if price and price <= target_per:
            affordable[code] = price
            print(f'  {code} 추가: {price:,}원 ✅')

# 상위 3개만 유지 (삼성전자, 삼성전자우, 카카오)
price_dict = {
    '005930': 299500,   # 삼성전자
    '005935': 188500,   # 삼성전자우
    '035720': 41100     # 카카오 (4등)
}

print('최종 편입 종목 3개:')
for code, price in price_dict.items():
    qty = 1_000_000 // price
    amt = qty * price
    print(f'  {code}: {price:,}원 × {qty}주 = {amt:,}원')

매수 후보 3개 종목 현재가 조회
        티커 |             종목명 |        현재가 |     등락률
  --------------------------------------------------
    005930 |          005930 |    299,250 | ▲ 2.31%
    009150 |          009150 |  1,560,000 | ▲16.42%
    005935 |          005935 |    188,500 | ▲ 0.53%

종목당 투자금: 1,000,000원

살 수 있는 종목:
  005930: 299,250원 ✅
  009150: 1,560,000원 ❌ (초과)
  005935: 188,500원 ✅

추가로 1개 종목 필요
  035720 추가: 41,100원 ✅
  003490 추가: 27,400원 ✅
  066970 추가: 158,800원 ✅
  028050 추가: 53,600원 ✅
  034730 추가: 649,000원 ✅
  011200 추가: 20,250원 ✅
  006400 추가: 635,000원 ✅
  032830 추가: 350,500원 ✅
  161390 추가: 63,100원 ✅
  000880 추가: 142,300원 ✅
  005387 추가: 272,500원 ✅
  267250 추가: 287,500원 ✅
  034020 추가: 111,100원 ✅
  032640 추가: 15,320원 ✅
  003550 추가: 122,400원 ✅
최종 편입 종목 3개:
  005930: 299,500원 × 3주 = 898,500원
  005935: 188,500원 × 5주 = 942,500원
  035720: 41,100원 × 24주 = 986,400원


---
## Cell 5 — 매수/매도 주문

> **⚠️ 실제 주문이 나갑니다**  
> TEST_MODE = True로 먼저 확인  
> 장 시간: 09:00 ~ 15:30 (한국 시간)

In [14]:
def place_order(code, qty, order_type='buy',
                 price=0, order_dvsn='01'):
    """
    국내 주식 주문

    Args:
        code      : 종목코드 (예: '005930')
        qty       : 주문 수량
        order_type: 'buy' 또는 'sell'
        price     : 주문 가격 (0이면 시장가)
        order_dvsn: '01'=시장가, '00'=지정가

    Returns:
        주문번호 또는 None
    """
    # 거래 ID
    # TTTC0802U: 매수
    # TTTC0801U: 매도
    tr_id = 'TTTC0802U' if order_type == 'buy' else 'TTTC0801U'

    url  = f'{BASE_URL}/uapi/domestic-stock/v1/trading/order-cash'
    body = {
        'CANO'        : ACCOUNT_NO,
        'ACNT_PRDT_CD': ACCOUNT_CD,
        'PDNO'        : code,
        'ORD_DVSN'    : order_dvsn,  # 01: 시장가
        'ORD_QTY'     : str(qty),
        'ORD_UNPR'    : str(price) if price > 0 else '0'
    }

    resp = requests.post(
        url,
        headers=get_headers(tr_id),
        json=body
    )

    if resp.status_code == 200:
        data = resp.json()
        if data['rt_cd'] == '0':
            order_no  = data['output']['KRX_FWDG_ORD_ORGNO']
            action_kr = '매수' if order_type == 'buy' else '매도'
            print(f'  ✅ {action_kr} 주문 성공: '
                  f'{code} {qty}주 (주문번호: {order_no})')
            return order_no
        else:
            print(f'  ❌ 주문 실패: {data["msg1"]}')
    else:
        print(f'  HTTP 오류: {resp.status_code}')
    return None


print('주문 함수 정의 완료')
print('실제 주문은 Cell 6에서 진행합니다')

주문 함수 정의 완료
실제 주문은 Cell 6에서 진행합니다


---
## Cell 6 — 목표 포트폴리오 계산 & 리밸런싱

In [23]:
def execute_rebalancing(target_df, current_holdings,
                         test_mode=True):
    """리밸런싱 실행"""
    mode = '[테스트]' if test_mode else '[실전]'
    now  = datetime.datetime.now()

    print(f'{mode} 리밸런싱 시작')
    print(f'실행 시간: {now.strftime("%Y-%m-%d %H:%M:%S")}')

    target_codes = set(target_df['code'].tolist())
    sell_orders  = []
    buy_orders   = []

    # 매도: 목표에 없는 보유 종목
    for code, qty in current_holdings.items():
        if code not in target_codes and qty > 0:
            sell_orders.append({'code': code, 'qty': qty})

    # 매수: 목표 포트폴리오
    for _, row in target_df.iterrows():
        code      = row['code']
        target_qty = int(row['qty'])
        cur_qty   = current_holdings.get(code, 0)
        diff      = target_qty - cur_qty

        if diff > 0:
            buy_orders.append({
                'code': code, 'qty': diff,
                'name': row['name'],
                'price': row['price']
            })
        elif diff < 0:
            sell_orders.append({
                'code': code, 'qty': abs(diff),
                'name': row['name']
            })

    # 주문 내역 출력
    print(f'\n매도 주문 ({len(sell_orders)}건):')
    if sell_orders:
        for o in sell_orders:
            price = price_dict.get(o['code'], 0)
            amt   = price * o['qty']
            print(f'  매도: {o["code"]} {o["qty"]}주 '
                  f'(약 {amt:,}원)')
    else:
        print('  없음')

    print(f'\n매수 주문 ({len(buy_orders)}건):')
    if buy_orders:
        for o in buy_orders:
            amt = o['price'] * o['qty']
            print(f'  매수: {o["code"]} {o.get("name","")} '
                  f'{o["qty"]}주 (약 {amt:,}원)')
    else:
        print('  없음')

    if not sell_orders and not buy_orders:
        print('\n리밸런싱 불필요')
        return

    # 실제 주문
    if not test_mode:
        # 장 시간 체크
        is_open = (
            now.weekday() < 5 and
            datetime.time(9, 0) <= now.time() <= datetime.time(15, 30)
        )
        if not is_open:
            print('\n⚠️  장 시간이 아닙니다 (09:00~15:30)')
            return

        print('\n주문 실행 중...')
        for o in sell_orders:
            place_order(o['code'], o['qty'],
                        order_type='sell', order_dvsn='01')
            time.sleep(0.5)

        if sell_orders:
            time.sleep(3)

        for o in buy_orders:
            place_order(o['code'], o['qty'],
                        order_type='buy', order_dvsn='01')
            time.sleep(0.5)

        print('\n✅ 리밸런싱 완료!')
    else:
        print(f'\n{mode} 실제 주문 없음')
        print('실전: TEST_MODE = False 로 변경')


# ── 목표 포트폴리오 설정 ──────────────────────────────────
price_dict = {
    '005930': 299250,
    '005935': 188500,
    '035720': 41100
}

target_data = [
    {'code': '005930', 'name': '삼성전자',   'price': 299250, 'qty': 3,  'amount': 897750},
    {'code': '005935', 'name': '삼성전자우', 'price': 188500, 'qty': 5,  'amount': 942500},
    {'code': '035720', 'name': '카카오',     'price': 41100,  'qty': 24, 'amount': 986400},
]
target_df = pd.DataFrame(target_data)

print('목표 포트폴리오:')
print(f'  {"티커":>8} | {"종목명":>10} | {"현재가":>10} | '
      f'{"수량":>6} | {"투자금액":>12}')
print('  ' + '-' * 58)
total = 0
for _, row in target_df.iterrows():
    print(f'  {row["code"]:>8} | {row["name"]:>10} | '
          f'{row["price"]:>10,} | {row["qty"]:>6}주 | '
          f'{row["amount"]:>12,}원')
    total += row['amount']
print(f'\n  총 투자금액: {total:,}원')
print(f'  잔여 현금  : {INVEST_AMOUNT - total:,}원')
print()

# 테스트 모드로 먼저 확인
TEST_MODE = False  # ← False로 바꾸면 실제 주문

execute_rebalancing(target_df, current_holdings,
                     test_mode=TEST_MODE)

목표 포트폴리오:
        티커 |        종목명 |        현재가 |     수량 |         투자금액
  ----------------------------------------------------------
    005930 |       삼성전자 |    299,250 |      3주 |      897,750원
    005935 |      삼성전자우 |    188,500 |      5주 |      942,500원
    035720 |        카카오 |     41,100 |     24주 |      986,400원

  총 투자금액: 2,826,650원
  잔여 현금  : 173,350원

[실전] 리밸런싱 시작
실행 시간: 2026-05-26 10:43:19

매도 주문 (0건):
  없음

매수 주문 (3건):
  매수: 005930 삼성전자 3주 (약 897,750원)
  매수: 005935 삼성전자우 5주 (약 942,500원)
  매수: 035720 카카오 24주 (약 986,400원)

주문 실행 중...
  ✅ 매수 주문 성공: 005930 3주 (주문번호: 91254)
  ✅ 매수 주문 성공: 005935 5주 (주문번호: 91254)
  ✅ 매수 주문 성공: 035720 24주 (주문번호: 91254)

✅ 리밸런싱 완료!


In [25]:
def execute_rebalancing(target_df, current_holdings,
                         test_mode=True):
    """
    리밸런싱 실행

    test_mode=True : 주문 내역만 출력
    test_mode=False: 실제 주문 실행
    """
    mode = '[테스트]' if test_mode else '[실전]'
    now  = datetime.datetime.now()

    print(f'{mode} 리밸런싱 시작')
    print(f'실행 시간: {now.strftime("%Y-%m-%d %H:%M:%S")}')

    # 장 시간 체크 (09:00 ~ 15:30)
    if not test_mode:
        is_market_open = (
            now.weekday() < 5 and  # 월~금
            datetime.time(9, 0) <= now.time() <= datetime.time(15, 30)
        )
        if not is_market_open:
            print('⚠️  장 시간이 아닙니다 (09:00~15:30)')
            print('장 시간에 다시 실행하세요')
            return

    target_codes = set(target_df['code'].tolist())
    sell_orders  = []
    buy_orders   = []

    # 매도: 목표에 없는 보유 종목
    for code, qty in current_holdings.items():
        if code not in target_codes and qty > 0:
            sell_orders.append({'code': code, 'qty': qty})

    # 매수: 목표 포트폴리오
    for _, row in target_df.iterrows():
        code       = row['code']
        target_qty = int(row['qty'])
        cur_qty    = current_holdings.get(code, 0)
        diff       = target_qty - cur_qty

        if diff > 0:
            buy_orders.append({
                'code': code, 'qty': diff,
                'name': row['name']
            })
        elif diff < 0:
            sell_orders.append({
                'code': code, 'qty': abs(diff),
                'name': row['name']
            })

    # 주문 내역 출력
    print(f'\n매도 주문 ({len(sell_orders)}건):')
    for o in sell_orders:
        price = price_dict.get(o['code'], 0)
        amt   = price * o['qty']
        print(f'  매도: {o["code"]} {o["qty"]}주 '
              f'(약 {amt:,}원)')

    print(f'\n매수 주문 ({len(buy_orders)}건):')
    for o in buy_orders:
        price = price_dict.get(o['code'], 0)
        amt   = price * o['qty']
        print(f'  매수: {o["code"]} {o.get("name","")} '
              f'{o["qty"]}주 (약 {amt:,}원)')

    if not sell_orders and not buy_orders:
        print('\n리밸런싱 불필요 — 이미 목표 포트폴리오와 동일')
        return

    # 실제 주문 실행
    if not test_mode:
        print('\n주문 실행 중...')

        # 매도 먼저 (현금 확보)
        for o in sell_orders:
            place_order(o['code'], o['qty'],
                        order_type='sell', order_dvsn='01')
            time.sleep(0.5)

        # 잠시 대기
        if sell_orders:
            print('매도 후 3초 대기...')
            time.sleep(3)

        # 매수
        for o in buy_orders:
            place_order(o['code'], o['qty'],
                        order_type='buy', order_dvsn='01')
            time.sleep(0.5)

        print('\n✅ 리밸런싱 완료!')
    else:
        print(f'\n{mode} 실제 주문 없음')
        print('실전 실행: execute_rebalancing(..., test_mode=False)')


# 테스트 모드로 먼저 확인
TEST_MODE = False  # ← False로 바꾸면 실제 주문

execute_rebalancing(
    target_df,
    current_holdings,
    test_mode=TEST_MODE
)

[실전] 리밸런싱 시작
실행 시간: 2026-05-26 10:43:42

매도 주문 (0건):

매수 주문 (3건):
  매수: 005930 삼성전자 3주 (약 897,750원)
  매수: 005935 삼성전자우 5주 (약 942,500원)
  매수: 035720 카카오 24주 (약 986,400원)

주문 실행 중...
  ❌ 주문 실패: 주문가능금액을 초과 했습니다
  ❌ 주문 실패: 주문가능금액을 초과 했습니다
  ❌ 주문 실패: 주문가능금액을 초과 했습니다

✅ 리밸런싱 완료!


---
## Cell 7 — 일일 모니터링

> **매일 실행할 것들:**
> 1. 잔고 확인
> 2. 보유 종목 손익 확인
> 3. 월말이면 자동 리밸런싱

In [26]:
def daily_monitoring(test_mode=True):
    """
    일일 모니터링 & 자동 리밸런싱
    매일 장 마감 후 실행 (15:30 이후)
    """
    now = datetime.datetime.now()
    print('=' * 50)
    print(f' 일일 모니터링: {now.strftime("%Y-%m-%d %H:%M")}')
    print('=' * 50)

    # 토큰 갱신
    global ACCESS_TOKEN
    ACCESS_TOKEN = get_access_token()

    # 잔고 조회
    deposit, total, holdings = get_balance()

    # 오늘 수익률 계산 (간략)
    print(f'\n총 평가금액: {total:,}원')

    # 월말 판단
    next_day      = now + datetime.timedelta(days=1)
    is_month_end  = (now.month != next_day.month)
    is_weekday    = now.weekday() < 5

    if is_month_end and is_weekday:
        print('\n📅 월말 — 리밸런싱 실행')

        # 최신 신호 업데이트 필요
        print('최신 신호 로드 중...')
        signal = pd.read_csv('data/processed/kr_latest_signal.csv')
        top_n  = signal.nlargest(N_STOCKS, 'score')

        # 현재가 조회
        price_dict_new = {}
        for _, row in top_n.iterrows():
            p, _, _, _ = get_price(row['ticker'])
            if p:
                price_dict_new[row['ticker']] = p
            time.sleep(0.1)

        # 목표 포트폴리오 계산
        target = calculate_target_portfolio(
            invest_amount=INVEST_AMOUNT,
            n_stocks=N_STOCKS,
            price_dict=price_dict_new
        )

        # 리밸런싱 실행
        execute_rebalancing(target, holdings,
                             test_mode=test_mode)
    else:
        days_left = (next_day.replace(day=1) - next_day).days
        print(f'\n다음 리밸런싱까지: 약 {abs(days_left)}일')
        print('(월말에 자동 리밸런싱 실행)')

    print('\n일일 모니터링 완료')


# 지금 바로 실행해보기
daily_monitoring(test_mode=True)

 일일 모니터링: 2026-05-26 10:45
토큰 발급 성공: eyJ0eXAiOiJKV1QiLCJh...
계좌 잔고 조회 성공
  예수금 (현금)  :       3,254,398원
  총 평가금액    :       3,262,348원

  보유 종목 (4개):
        티커 |          종목명 |     수량 |       평균단가 |        현재가 |     손익률
  -----------------------------------------------------------------
    005930 |         삼성전자 |      3 |    299,750 |    300,000 |   0.08%
    005935 |        삼성전자우 |      5 |    188,300 |    188,300 |   0.00%
    035720 |          카카오 |     24 |     41,100 |     41,200 |   0.24%
    073240 |        금호타이어 |      0 |          0 |      4,910 |   0.00%

총 평가금액: 3,262,348원

다음 리밸런싱까지: 약 26일
(월말에 자동 리밸런싱 실행)

일일 모니터링 완료


---
## Cell 8 — 스케줄러 (자동 실행)

> **이 셀을 실행하면:**
> 매일 오전 9시: 잔고 확인
> 매일 오후 4시: 일일 모니터링 + 월말이면 리밸런싱
> 주피터 커널이 살아있는 동안 계속 실행됨

In [ ]:
try:
    from apscheduler.schedulers.background import BackgroundScheduler

    scheduler = BackgroundScheduler()

    # 매일 오전 9시: 장 시작 전 잔고 확인
    scheduler.add_job(
        lambda: get_balance(),
        'cron', hour=9, minute=0,
        id='morning_check'
    )

    # 매일 오후 4시: 장 마감 후 모니터링
    scheduler.add_job(
        lambda: daily_monitoring(test_mode=True),
        'cron', hour=16, minute=0,
        id='daily_monitoring'
    )

    print('스케줄러 설정 완료')
    print('  09:00 — 잔고 확인')
    print('  16:00 — 일일 모니터링 (월말이면 리밸런싱)')
    print()
    print('실전 자동매매 시작하려면:')
    print('  1. test_mode=True → False 변경')
    print('  2. scheduler.start() 실행')
    print('  3. 커널 종료하지 말것')
    print()

    # scheduler.start()  # ← 주석 해제하면 자동 실행 시작

except ImportError:
    print('pip install apscheduler 먼저 실행하세요')

---
## Cell 9 — 실전 체크리스트

실전 투입 전에 반드시 확인하세요:

**기술적 체크:**
- [ ] 토큰 발급 성공 (Cell 2)
- [ ] 잔고 조회 성공 (Cell 3)
- [ ] 현재가 조회 성공 (Cell 4)
- [ ] 테스트 모드 주문 내역 확인 (Cell 6)

**자금 체크:**
- [ ] 계좌에 100만원 입금 확인
- [ ] 예수금이 100만원 이상인지 확인

**전략 체크:**
- [ ] 최신 신호 종목 20개 확인
- [ ] 목표 포트폴리오 종목당 금액 확인
- [ ] 장 시간(09:00~15:30)에 실행 예정인지 확인

**실전 시작:**
```python
# Cell 6에서
TEST_MODE = False  # 이걸로 바꾸고 실행
```

In [ ]:
# 최종 요약 출력
print('=' * 50)
print(' 실전 투입 준비 상태')
print('=' * 50)
print(f'  투자 금액    : {INVEST_AMOUNT:,}원')
print(f'  편입 종목 수  : {N_STOCKS}개')
print(f'  종목당 금액   : {INVEST_AMOUNT//N_STOCKS:,}원')
print()
print('  매수 후보 종목:')
for rank, (_, row) in enumerate(
    pd.read_csv('data/processed/kr_latest_signal.csv')
    .nlargest(N_STOCKS, 'score').iterrows(), 1
):
    price = price_dict.get(row['ticker'], 0)
    qty   = (INVEST_AMOUNT // N_STOCKS) // price if price > 0 else 0
    amt   = qty * price
    print(f'  {rank:>2}. {row["ticker"]} '
          f'{qty}주 ({amt:,}원)')
print()
print('준비되면 Cell 6에서 TEST_MODE = False로 변경 후 실행')